In [6]:
import pandas as pd 

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

y_train_cv = pd.read_csv("../data/processed/y_train.csv").squeeze()
print("number of folds :", cv.get_n_splits())
print("training samples: ", len(y_train_cv))
print(y_train_cv.value_counts())
print(y_train_cv.value_counts(normalize=True).round(3))

number of folds : 5
training samples:  5634
Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [8]:
X_train = pd.read_csv("../data/processed/X_train_cluster.csv")
X_test = pd.read_csv("../data/processed/X_test_cluster.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")

In [15]:

models = {
    "Logistic Regression" : LogisticRegression(
        random_state=42,
        max_iter=5000,
        solver="lbfgs"
    ),
    "SVM": SVC(
        random_state=42,
    ),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=1.0
    )

}   
results =  []

trained_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])
    scores = cross_validate(
        pipeline,
        X_train,
        y_train_cv,
        cv=cv,
        scoring={
            "f1": "f1",
            "roc_auc": "roc_auc",
            "recall": "recall"
        },
        return_train_score=False,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),
        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std(),
        "Recall Mean": scores["test_recall"].mean(),
        "Recall Std": scores["test_recall"].std()
    })

cv_results = (
    pd.DataFrame(results)
    .sort_values("F1 Mean", ascending=False)
    .reset_index(drop=True)
)

print(cv_results.round(4).to_string(index=False))

/home/alaric/Desktop/studies/projects/client-churn-prediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              Model  F1 Mean  F1 Std  ROC-AUC Mean  ROC-AUC Std  Recall Mean  Recall Std
Logistic Regression   0.6388  0.0257        0.8441       0.0122       0.7258      0.0368
            XGBoost   0.5442  0.0271        0.8168       0.0085       0.5010      0.0322
                SVM   0.3616  0.0140        0.5139       0.0049       0.5324      0.0576
